# Coverage of a Mars geological feature

Pick a feature, confirm, and see what every instrument has observed of it.

- **Per-observation coverage** shows what a single observation covered, as a share of the
  feature's bounding box, at the time it was taken.
- **Pixels per observation** shows how many pixels one observation lands, instrument
  by instrument, each of them read on a scale of its own.
- **Cumulative coverage** shows how much of the feature each instrument has reached in
  total, as its observations accumulate.
- **Candidate windows** shows every stretch of time one tile of the feature could be
  studied over, and how much of that tile each of them reaches.

### What is measured

**Feature's bounding box:** The ODE catalogue publishes a
south/north/west/east box per feature and not its true outline, so the box is what
coverage is measured against.

**The ground is flattened:** A degree is not a fixed amount of ground: near the poles it covers far less than at the equator. So the same patch measured in degrees would look bigger or smaller depending only on where it happens to sit. Everything is therefore laid onto a flat map drawn for that one place, of the kind that keeps every patch its true size wherever it falls. Areas can then just be added up, and a share means the same thing at any latitude.

**CTX and CRISM publish areas.** ODE gives each of these products a footprint
polygon, so the observation is taken as published: cut to the feature box in lon/lat,
resampled and projected. The resolution of the product is not considered, just the area it covers.

**MOLA excluded.** Since it's coverage is full planet, it is not useful to show it here.

**SHARAD publishes a bare line.** It is a sounder, so ODE gives the ground track it flew and no width at all. A width is needed before a line can become area, and rather than pick one number for the whole mission, every track derives its own: its published length divided by its duration gives the ground speed, that speed gives the altitude a circular orbit would need to fly it, and the altitude gives the first Fresnel zone, the strip a nadir sounder illuminates. It lands near 3 km. Only published constants and the track's own numbers go in; nothing is fitted to the data.
A track that will not solve, usually because no stop time was published, takes the median width of the other tracks in the same feature, or a nominal width if none of them solved. Every row records the width it used and whether it was derived or fallen back on, so a guess can always be told from a measurement.


### What is left out

**Only targeted sets with both a footprint and a time are downloaded:** An observation
with no acquisition time cannot be placed on a time axis at all, and a whole-planet
basemap reads full coverage on every feature by construction while saying nothing about
targeted observing. Neither kind is downloaded, computed, or plotted here.

**Records that cannot be used are counted, never dropped quietly:** A record needs a
footprint and a start time. The stop time is not required: it only refines a track's swath
width, which already falls back, so demanding it would throw away a whole observation to
avoid guessing one number. An instrument set whose records are all unusable is reported as
having measured nothing, with its discard count, rather than passing for a set that had
nothing to say.

**An observation is kept when it overlaps at all:** A footprint that only partly reaches
into the feature is cropped to it and counted for the part that reaches. Only an
observation with no overlap whatsoever is discarded.

### Setup

In [ ]:
"""Import the pipeline's own pickers, plots, and tables."""

from visualization.common.picker import FeaturePicker
from visualization.feature.picker import TilePicker
from visualization.feature.plots import (
    basemap,
    cumulative,
    density,
    footprints,
    observations,
    pixels,
    windows,
)
from visualization.feature.tables import held, shortfall, summary

## Select a feature

Feature types and names that have no local data are marked. Confirming one of those shows
the grey panel instead of plots, so it is always clear whether a missing line means "not
observed" or "not downloaded". Every cell below claims an area rather than reading a choice, so running the whole
notebook is safe: nothing draws until you confirm, and confirming a different feature
refills the areas without rerunning anything.

The **strategy dropdown** picks what a window has to hold. Every panel below is drawn
under the one chosen, and changing it redraws them without reloading the feature.

The mosaic drawn under every feature is THEMIS daytime infrared, served as WMS by the
USGS.

In [ ]:
"""Show the feature picker, which fills every area claimed below it."""

picker = FeaturePicker()
picker.choose()

### Load the confirmed feature's coverage

The mosaic is overlaid with the grid of tiles the search cuts the feature into, drawn
where the search really cuts it. A tile is outlined green when it earned a window and
red when the search refused it, and a tile holding none of the feature is not drawn at
all. The overlay is there to show how the ground is treated, not to be measured off.

In [ ]:
"""Report the feature and show how the search cuts it into tiles."""

picker.show_panel(basemap.plot)

## The feature as a whole

Everything in this section is measured over the whole feature, ignoring the tile grid used by the search algorithm. 

### Per-observation coverage

One point per observation, at the time it was taken, each dropped to the axis by a thin
stem so a single observation stays visible where the points crowd together. The height is
the share of the feature's bounding box that one observation covered, so a tall point is a
wide swath and a low point is a narrow one.

Zero-coverage points are plotted since they are still observations taken in consideration by ODE, however the intersection is minimal.

In [ ]:
"""Plot what each single observation covered, one panel per instrument."""

picker.show_panel(observations.plot)

### Density of acquisitions

Each line with a monthly granularity shows based on the number of observations taken in that month, the density of acquisitions for each instrument. 

In [ ]:
"""Plot when each instrument was busy on the feature, one row per set."""

picker.show_panel(density.plot)

### Cumulative coverage

It shows how much of the feature each instrument has reached in total, as its observations accumulate through time. 

In [ ]:
"""Plot how much of the feature each instrument reaches over time, and in total."""

picker.show_panel(cumulative.plot)

### Pixels per observation

The same observations as above, read in what they actually deliver: how many
pixels one of them lands inside the feature. Every instrument is drawn on a log
scale of its own, since CTX lands tens of millions of pixels where SHARAD lands a
few hundred traces, and one axis for both would flatten either into a wall at its
edge. The dashed line is the middle observation of its own panel.


In [ ]:
"""Plot how many pixels one observation lands, one panel per instrument."""

picker.show_panel(pixels.plot)

### Across its tiles

The search runs a tile at a time, so what the feature holds is what its tiles hold
between them. This tables summarizes the overall tiles situation of the feature.

In [ ]:
"""Summarise what the feature holds across the tiles the search ran over."""

picker.show_panel(summary.plot)

## Tile by tile

A tile is judged on its own, so an observation clipping its edge is left off here as it is left off the search, and the shares are shares of that
tile rather than of the feature.

In [ ]:
"""Show the tile picker, which fills every area claimed below it."""

tiles = TilePicker(picker)
tiles.choose()

### Candidate windows

Every stretch of time the tile's record could be clustered into, by when it is centred
and how long it runs. The colour is the share of the tile the window reaches, counted
evenly over the instruments that observed it, so a window one of them misses reads as
the poor window it is. A ring encloses the windows two instruments observed inside,
then three, and so on, drawn solid where it holds every one of them. Grey is a window
no sounder track passes through, since it's the main instrument that provides swath depth.

In [ ]:
"""Plot every window the tile could be studied over, and the one picked."""

tiles.show_panel(windows.plot)

### Per-observation coverage

The same panel as above the tiles, over this one tile alone. The stretch the window is
open over is shaded across it, except on the panel of an instrument the strategy asks
of the whole record, which no window binds.

In [ ]:
"""Plot what each single observation covered of the tile."""

tiles.show_panel(observations.plot_tile)

### Pixels the tile is offered

The same panels over this one tile, each observation counted for the part of its
footprint the tile holds. This is the count the strategy's pixel bar is read
against, so only the looks the tile admitted are drawn: one landing under that bar
never reaches the search at all.


In [ ]:
"""Plot how many pixels one observation lands on the tile."""

tiles.show_panel(pixels.plot_tile)

### What the tile holds

The tile itself, the window it earned, and what each instrument left on it in ground
and in the pixels that ground is worth. An instrument the strategy asks of the whole
record answers from wherever on the record it flew, so what it brought the tile is
counted here even where the window does not reach it.

In [ ]:
"""Summarise what the search left on the tile."""

tiles.show_panel(held.plot)

### Why it earned no window

The same tile searched again with the ground bars lifted, so a tile the
search refused still comes back with the window it came closest with. Each
instrument is set against what the strategy asks of it: the share it reaches
inside that window, and the most it ever reaches over the whole record. An
instrument short in the window but not over the record could cover the tile,
just never at the same time as the others; one short over the record could
not cover it whenever it flew.

In [ ]:
"""Report what the tile could bring when no ground is asked of it."""

tiles.show_panel(shortfall.plot)

### The footprints it keeps

The tile's own crop of the mosaic, with the footprint of every observation it keeps
traced on it, one colour per instrument set. The footprints are the ones ODE published,
so a sounder is drawn as the bare track it flew rather than as the swath the
measurement widened it into.

In [ ]:
"""Map the tile with the footprint of every observation its window keeps."""

tiles.show_panel(footprints.plot)